In [11]:
import pandas as pd
import os
from sklearn.linear_model import Ridge
from soupsieve import select

from src.evaluate import evaluate_model
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.svm import SVR, LinearSVR
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import matplotlib.pyplot as plt
import seaborn as sns
import shap
from sklearn.model_selection import cross_val_score
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from lightgbm import LGBMRegressor
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import KBinsDiscretizer
import shap
from scipy.stats import spearmanr
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor, StackingRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, mean_absolute_error
from xgboost import XGBRegressor
from sklearn.model_selection import cross_val_score
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error, mean_absolute_error
import xgboost as xgb
from sklearn.ensemble import RandomForestRegressor
import lightgbm as lgb


In [2]:
print(os.getcwd())
os.chdir("../")
os.getcwd()

/Users/antoninbenard/PycharmProjects/PythonProject3


'/Users/antoninbenard/PycharmProjects'

In [3]:
df_train = pd.read_parquet("PythonProject3/data/idf_vf_train_selected_features.parquet")
df_test = pd.read_parquet("PythonProject3/data/idf_vf_test_selected_features.parquet")
df_selected_oos = pd.read_parquet("PythonProject3/data/idf_vf_out_of_sample_selected_features (3).parquet")

In [4]:
df_train.shape, df_test.shape, df_selected_oos.shape

((507388, 24), (126848, 24), (47768, 24))

## Xgboost

A modifier pour prendre avec ou sans Paris

In [5]:
df_train = df_train[df_train["code_departement"] != "75"]
df_test = df_test[df_test["code_departement"] != "75"]
df_selected_oos = df_selected_oos[df_selected_oos["code_departement"] != "75"]

In [6]:
df_train = df_train.drop(columns="code_departement")
df_test = df_test.drop(columns="code_departement")
df_selected_oos = df_selected_oos.drop(columns="code_departement")

In [ ]:
X_train = df_train.drop(columns=['prix_m2'])
y_train = df_train['prix_m2']

X_test = df_test.drop(columns=['prix_m2'])
y_test = df_test['prix_m2']

X_oos = df_selected_oos.drop(columns=['prix_m2'])
y_oos = df_selected_oos['prix_m2']

In [ ]:
# Hyperparamètres trouvés avec Optuna
params = {
    'n_estimators': 775,
    'max_depth': 12,
    'learning_rate': 0.014782011673253295,
    'subsample': 0.9357174649094887,
    'colsample_bytree': 0.6147138328324339,
    'min_child_weight': 5,
    'gamma': 5.722492597986979,
    'reg_alpha': 0.05048493723008041,
    'reg_lambda': 2.3619571121802574
}

In [7]:
model = xgb.XGBRegressor(**params)
model.fit(X_train, y_train)

y_pred_train = model.predict(X_train)
y_pred_test = model.predict(X_test)
y_pred_oos = model.predict(X_oos)

In [8]:
def metrics(y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    return rmse, mae

train_rmse, train_mae = metrics(y_train, y_pred_train)
test_rmse, test_mae = metrics(y_test, y_pred_test)
oos_rmse, oos_mae = metrics(y_oos, y_pred_oos)

print(f"\nTrain - RMSE: {train_rmse:.2f} | MAE: {train_mae:.2f}")
print(f"Test  - RMSE: {test_rmse:.2f} | MAE: {test_mae:.2f}")
print(f"OOS   - RMSE: {oos_rmse:.2f} | MAE: {oos_mae:.2f}")



Train - RMSE: 1413.57 | MAE: 1036.57
Test  - RMSE: 2033.57 | MAE: 1489.73
OOS   - RMSE: 1529.98 | MAE: 1111.10


In [9]:
feature_importance = pd.DataFrame({
    'feature': X_train.columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

print("\nTop 15 features:")
print(feature_importance.head(15).to_string(index=False))


Top 15 features:
                                        feature  importance
                prix_m2_median_maisons_voisines    0.114011
                             ratio_terrain_bati    0.100392
commune_ratio_residences_secondaires_population    0.090489
                                 is_appartement    0.065870
                     commune_revenu_median_2020    0.057163
                 commune_part_commerce_tourisme    0.055341
           commune_education_score_2013_commune    0.049986
                              nb_services_2000m    0.045233
                       nb_transport_autre_2000m    0.038737
                                    nombre_lots    0.037448
          commune_etablissements_1_salarie_2021    0.034917
                   Taux_moyen_credit_immo_lag1t    0.034763
                     commune_taux_proprietaires    0.030023
    commune_densite_residences_principales_2020    0.029235
                              densite_rel_2000m    0.029078


## Stacking RF + Xgboost (les résultats sont nuls)

In [ ]:
RANDOM_STATE = 42
SAMPLE_PCT = 0.4
TARGET_COL = 'prix_m2'
CV_FOLDS = 5

# Hyperparamètres fixes (pré-optimisés)
XGB_PARAMS = {
    'n_estimators': 775,
    'max_depth': 12,
    'learning_rate': 0.014782011673253295,
    'subsample': 0.9357174649094887,
    'colsample_bytree': 0.6147138328324339,
    'min_child_weight': 5,
    'gamma': 5.722492597986979,
    'reg_alpha': 0.05048493723008041,
    'reg_lambda': 2.3619571121802574,
    'random_state': RANDOM_STATE,
    'n_jobs': -1,
    'verbosity': 0
}

RF_PARAMS = {
    'n_estimators': 278,
    'max_depth': 17,
    'min_samples_split': 3,
    'min_samples_leaf': 1,
    'max_features': 0.35739984068330766,
    'random_state': RANDOM_STATE,
    'n_jobs': -1
}

In [ ]:
X_train = df_train.drop(columns=[TARGET_COL])
y_train = df_train[TARGET_COL]

X_test = df_test.drop(columns=[TARGET_COL])
y_test = df_test[TARGET_COL]

X_oos = df_selected_oos.drop(columns=[TARGET_COL], errors='ignore')
if TARGET_COL in df_selected_oos.columns:
    y_oos = df_selected_oos[TARGET_COL]
    has_oos_target = True
else:
    y_oos = None
    has_oos_target = False

In [7]:
if SAMPLE_PCT < 1.0:
    from sklearn.preprocessing import KBinsDiscretizer
    n_bins = 10
    binner = KBinsDiscretizer(n_bins=n_bins, encode='ordinal', strategy='quantile')
    y_train_binned = binner.fit_transform(y_train.values.reshape(-1, 1)).ravel().astype(int)

    sample_size = int(len(X_train) * SAMPLE_PCT)
    sample_indices = []

    for bin_val in range(n_bins):
        bin_indices = np.where(y_train_binned == bin_val)[0]
        n_samples_bin = int(sample_size / n_bins)
        if len(bin_indices) > 0:
            sampled = np.random.choice(
                bin_indices,
                size=min(n_samples_bin, len(bin_indices)),
                replace=False
            )
            sample_indices.extend(sampled)

    X_train_sample = X_train.iloc[sample_indices].reset_index(drop=True)
    y_train_sample = y_train.iloc[sample_indices].reset_index(drop=True)

    print(f"\nÉchantillon créé: {len(X_train_sample):,} lignes")

    xgb_sample = XGBRegressor(**XGB_PARAMS)
    rf_sample = RandomForestRegressor(**RF_PARAMS)

    estimators_sample = [('xgb', xgb_sample), ('rf', rf_sample)]

    stacking_ridge_sample = StackingRegressor(
        estimators=estimators_sample,
        final_estimator=Ridge(alpha=1.0),
        cv=CV_FOLDS,
        n_jobs=-1
    )
    stacking_ridge_sample.fit(X_train_sample, y_train_sample)

    stacking_rf_sample = StackingRegressor(
        estimators=estimators_sample,
        final_estimator=RandomForestRegressor(
            n_estimators=100,
            max_depth=10,
            random_state=RANDOM_STATE,
            n_jobs=-1
        ),
        cv=CV_FOLDS,
        n_jobs=-1
    )
    stacking_rf_sample.fit(X_train_sample, y_train_sample)


    models_sample = {
        'Stacking Ridge': stacking_ridge_sample,
        'Stacking RF': stacking_rf_sample
    }

    results_sample = []

    for name, model in models_sample.items():
        y_train_pred = model.predict(X_train_sample)
        y_test_pred = model.predict(X_test)

        train_rmse = np.sqrt(mean_squared_error(y_train_sample, y_train_pred))
        train_mae = mean_absolute_error(y_train_sample, y_train_pred)

        test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
        test_mae = mean_absolute_error(y_test, y_test_pred)

        print(f"\n{name}:")
        print(f"  Train (sample) - RMSE: {train_rmse:>8.2f} | MAE: {train_mae:>8.2f}")
        print(f"  Test           - RMSE: {test_rmse:>8.2f} | MAE: {test_mae:>8.2f}")

        # OOS
        if has_oos_target:
            y_oos_pred = model.predict(X_oos)
            oos_rmse = np.sqrt(mean_squared_error(y_oos, y_oos_pred))
            oos_mae = mean_absolute_error(y_oos, y_oos_pred)
            print(f"  OOS            - RMSE: {oos_rmse:>8.2f} | MAE: {oos_mae:>8.2f}")
            results_sample.append([name, train_rmse, train_mae, test_rmse, test_mae, oos_rmse, oos_mae])
        else:
            results_sample.append([name, train_rmse, train_mae, test_rmse, test_mae, np.nan, np.nan])

    cols = ['Modèle', 'Train RMSE', 'Train MAE', 'Test RMSE', 'Test MAE', 'OOS RMSE', 'OOS MAE']
    if not has_oos_target:
        cols = cols[:-2]
        results_sample = [r[:-2] for r in results_sample]

    df_results_sample = pd.DataFrame(results_sample, columns=cols)
    print(f"\n{df_results_sample.to_string(index=False)}")


Dimensions des datasets:
  Train: 409,591 lignes × 22 features
  Test:  102,350 lignes × 22 features
  OOS:   37,819 lignes × 22 features

OPTION 1: TEST RAPIDE SUR ÉCHANTILLON (40%)

→ Échantillon créé: 163,830 lignes

→ Entraînement Stacking Ridge sur échantillon...


/Users/antoninbenard/PycharmProjects/PythonProject3/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_discretization.py:296: FutureWarning: The current default behavior, quantile_method='linear', will be changed to quantile_method='averaged_inverted_cdf' in scikit-learn version 1.9 to naturally support sample weight equivalence properties by default. Pass quantile_method='averaged_inverted_cdf' explicitly to silence this warning.
  warnings.warn(


→ Entraînement Stacking RF sur échantillon...

RÉSULTATS SUR ÉCHANTILLON (40%)

Stacking Ridge:
  Train (sample) - RMSE:   987.82 | MAE:   726.68
  Test           - RMSE:  1379.30 | MAE:   964.59
  OOS            - RMSE:  1267.34 | MAE:   884.45

Stacking RF:
  Train (sample) - RMSE:  2132.23 | MAE:  1433.75
  Test           - RMSE:  2162.86 | MAE:  1482.24
  OOS            - RMSE:  2022.45 | MAE:  1387.83

        Modèle  Train RMSE   Train MAE   Test RMSE    Test MAE    OOS RMSE     OOS MAE
Stacking Ridge  987.823383  726.678295 1379.296895  964.593760 1267.335802  884.452500
   Stacking RF 2132.229292 1433.750515 2162.861249 1482.242091 2022.448234 1387.830291

✓ Test sur échantillon terminé!
  Temps d'entraînement réduit grâce au sampling.
  Pour entraîner sur l'ensemble complet, changez SAMPLE_PCT = 1.0


## Random Forest

In [9]:
X_train = df_train.drop(columns=['prix_m2'])
y_train = df_train['prix_m2']

X_test = df_test.drop(columns=['prix_m2'])
y_test = df_test['prix_m2']

X_oos = df_selected_oos.drop(columns=['prix_m2'])
y_oos = df_selected_oos['prix_m2']

# Hyperparamètres Random Forest
rf_params = {
    'n_estimators': 278,
    'max_depth': 17,
    'min_samples_split': 3,
    'min_samples_leaf': 1,
    'max_features': 0.35739984068330766,
    'random_state': 42,
    'n_jobs': -1
}

model = RandomForestRegressor(**rf_params)
model.fit(X_train, y_train)

y_pred_train = model.predict(X_train)
y_pred_test = model.predict(X_test)
y_pred_oos = model.predict(X_oos)


In [10]:
def metrics(y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    return rmse, mae

train_rmse, train_mae = metrics(y_train, y_pred_train)
test_rmse, test_mae = metrics(y_test, y_pred_test)
oos_rmse, oos_mae = metrics(y_oos, y_pred_oos)

print(f"\nTrain - RMSE: {train_rmse:.2f} | MAE: {train_mae:.2f}")
print(f"Test  - RMSE: {test_rmse:.2f} | MAE: {test_mae:.2f}")
print(f"OOS   - RMSE: {oos_rmse:.2f} | MAE: {oos_mae:.2f}")


Train - RMSE: 968.70 | MAE: 657.91
Test  - RMSE: 1264.20 | MAE: 809.93
OOS   - RMSE: 1039.32 | MAE: 707.11


## LightGBM

In [12]:
X_train = df_train.drop(columns=['prix_m2'])
y_train = df_train['prix_m2']

X_test = df_test.drop(columns=['prix_m2'])
y_test = df_test['prix_m2']

X_oos = df_selected_oos.drop(columns=['prix_m2'])
y_oos = df_selected_oos['prix_m2']

In [ ]:
lgb_params = {
    'n_estimators': 278,
    'max_depth': 17,
    'learning_rate': 0.05,
    'num_leaves': 31,
    'min_child_samples': 20,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'random_state': 42
}

In [13]:
model = lgb.LGBMRegressor(**lgb_params)
model.fit(X_train, y_train)

y_pred_train = model.predict(X_train)
y_pred_test = model.predict(X_test)
y_pred_oos = model.predict(X_oos)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.004829 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3958
[LightGBM] [Info] Number of data points in the train set: 409591, number of used features: 22
[LightGBM] [Info] Start training from score 4814.259283


In [14]:
def metrics(y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    return rmse, mae

train_rmse, train_mae = metrics(y_train, y_pred_train)
test_rmse, test_mae = metrics(y_test, y_pred_test)
oos_rmse, oos_mae = metrics(y_oos, y_pred_oos)

print(f"\nTrain - RMSE: {train_rmse:.2f} | MAE: {train_mae:.2f}")
print(f"Test  - RMSE: {test_rmse:.2f} | MAE: {test_mae:.2f}")
print(f"OOS   - RMSE: {oos_rmse:.2f} | MAE: {oos_mae:.2f}")


Train - RMSE: 1337.01 | MAE: 853.93
Test  - RMSE: 1352.83 | MAE: 862.30
OOS   - RMSE: 1345.95 | MAE: 871.94
